# **<font color="red">ADK with Agent2Agent (A2A) Protocol</font>**
- With ADK, we can build complex multi-agent systems where different agents need to collaborate and interact using A2A Protocol.
- As you build more complex agentic systems, you will find that a single agent is often not enough.

## **<font color="blue">When to Use A2A vs. Local Sub-Agents</font>**
- **Local Sub-Agents:**
  - These are the agents that run _within the same application process_ as your main agent. They are like internal modules or libraries, used to organize your code into logical, reusable components.
  - Communication between a main agent and its local sub-agents is very fast because it happens directly in memory, without network overhead.
- **Remote Agents (A2A):**
  - These are independent agents that run as separate services, communicating over a network. A2A defines the standard protocol for this communication.
  - Consider using **A2A** when:
    - The agent you need to talk to is a __separate, standalone service_ (e.g., a specialized financial modeling agent).
    - The agent is maintained by a **different team or organization**.
    - You need to connect agents written in **different programming language or agent frameworks**.
    - You want to enforece a **strong, formal contract** (the A2A protocol) between your system's components.

### **<font color="green">When to Use A2A</font>**
- **Integrating with a Third-Party Service:** The provider exposes its data through an A2A-compatible agent.
- **Microservices Architecture:** You have a large system broken down into smaller independent services. A2A is ideal for these services to communicate with each other across network boundaries.
- **Cross-Language Communication:** Your core bussiness logic is in a Python agent, but you have a legacy system or a specialized component written in Java that you want to integrate as an agent. A2A provides the standardized communication layer.
- **Formal API Enforcement:** You are building a platform where different teams contribute agents, and you need to strict contract for how these agents interact to ensure compatibility and stability.

### **<font color="green">When NOT to Use A2A (Prefer Local Sub-Agents)</font>**
- **Internal Code Organization:** You are breaking down a complex task within a single agent into smaller, manageable functions or modules (e.g., a `DataValidator` sub-agent that cleans input data before processing). These are best handled as local sub-agents for performance and simplicity.
- **Performance-Critical Internal Operations:** A sub-agent is responsible for a high-frequency, low-latency operation that is tightly coupled with the main agent's execution (e.g., a `RealTimeAnalytics` sub-agent that processes data streams within the same application).
- **Shared Memory/Context:** When sub-agents need direct access to the main agent's internal state or shared memory for efficiency, A2A's network overhead and serialization/deserialization woould be counterproductive.
- **Simple Helper Functions:** For small, reusable pieces of logic that don't require independent deployment or complex state management, a simple function or class within the same agent is more appropriate than a separate A2A agent.


## **<font color="blue">The A2A Workflow in ADK: A Simple View</font>**
- ADK simplifies the process of building and connecting agents using the A2A protocol. Here's a straightforward breakdown of how it works:
  1. **Making an Agent Accessible (Exposing):**
     - You start with an existing ADK agent that you want other agents to be able to interact with.
     - The ADK provides a simple way to "expose" this agent, turning it into an **A2A Server**.
     - This server acts as a public interface, allowing other agents to send requests to your agent over a network. 
  2. **Connecting to an Accessible Agent (Consuming):**
     - In a separate agent (which could be running on the same machine or a different one), you'll use a special ADK component called `RemoteA2aAgent`.
     - This `RemoteA2aAgent` acts as a client that knows how to communicate with the **A2A Server** you exposed earlier.
     - It handles all the complexities of network communication, authentication, and data formatting behing the scenes.
- From your perspective as a developer, once you have set up this connection, interacting with the remote agent feels just like interacting with a local tool or function.
- The ADK abstracts away the network layer, making distributed agent systems as easy to work with as local ones.


## **<font color="blue">Visualizing the A2A Workflow</font>**
- To further clarify the A2A workflow, let's look at the "before and after" for both exposing and consuming agents, and then the combined system.

### **<font color="green">Exposing an Agent</font>**
- **Before Exposing:** Your agent code runs as a standalone component, but in this scenario, you want to expose it so that other remote agents can interact with your agent.

```text
+-------------------+
| Your Agent Code   |
|   (Standalone)    |
+-------------------+
```

- **After Exposing:** Your agent code is integrated with an `A2AServer` (an ADK component), making it accessible over a network to other remote agents.

```text
+-----------------+
|   A2A Server    |
| (ADK Component) |<--------+
+-----------------+         |
        |                   |
        v                   |
+-------------------+       |
| Your Agent Code   |       |
| (Now Accessible)  |       |
+-------------------+       |
                            |
                            | (Network Communication)
                            v
+-----------------------------+
|       Remote Agent(s)       |
|    (Can now communicate)    |
+-----------------------------+
```

### **<font color="green">Consuming an Agent</font>**
- **Before Consuming:** Your agent (referred to as the "Root Agent" in this context) is the application you are developing that needs to interact with a remote agent. Before consuming, it lacks the direct mechanism to do so.

```text
+----------------------+         +-------------------------------------------------------------+
|      Root Agent      |         |                        Remote Agent                         |
| (Your existing code) |         | (External Service that you want your Root Agent to talk to) |
+----------------------+         +-------------------------------------------------------------+
```
- **After Consuming:** Your Root Agent uses a `RemoteA2aAgent` (an ADK component that acts as a client-side proxy for the remote agent) to establish communication with the remote agent.

```text
+----------------------+         +-----------------------------------+
|      Root Agent      |         |         RemoteA2aAgent            |
| (Your existing code) |<------->|         (ADK Client Proxy)        |
+----------------------+         |                                   |
                                 |  +-----------------------------+  |
                                 |  |         Remote Agent        |  |
                                 |  |      (External Service)     |  |
                                 |  +-----------------------------+  |
                                 +-----------------------------------+
      (Now talks to remote agent via RemoteA2aAgent)
```

### **<font color="green">Final System (Combined View)</font>**
- This diagram shows how the consuming and exposing parts connect to form a complete A2A system.

```text
Consuming Side:
+----------------------+         +-----------------------------------+
|      Root Agent      |         |         RemoteA2aAgent            |
| (Your existing code) |<------->|         (ADK Client Proxy)        |
+----------------------+         |                                   |
                                 |  +-----------------------------+  |
                                 |  |         Remote Agent        |  |
                                 |  |      (External Service)     |  |
                                 |  +-----------------------------+  |
                                 +-----------------------------------+
                                                 |
                                                 | (Network Communication)
                                                 v
Exposing Side:
                                               +-----------------+
                                               |   A2A Server    |
                                               | (ADK Component) |
                                               +-----------------+
                                                       |
                                                       v
                                               +-------------------+
                                               | Your Agent Code   |
                                               | (Exposed Service) |
                                               +-------------------+
```



## **<font color="blue">Concrete Use Case: Customer Service and Product Catalog Agents</font>**
- Let's consider a practical example: a **Customer Service Agent** that needs to retrieve product information from a separate **Product Catalog Agent**.

### **<font color="green">Before A2A</font>**
- Initially, your Customer Service Agent might not have a direct, standardized way to query the Product Catalog Agent, especially if it's a separate service or managed by a different team.

```text
+-------------------------+         +--------------------------+
| Customer Service Agent  |         |  Product Catalog Agent   |
| (Needs Product Info)    |         | (Contains Product Data)  |
+-------------------------+         +--------------------------+
      (No direct, standardized communication)
```

### **<font color="green">After A2A</font>**
- By using the A2A Protocol, the Product Catalog Agent can expose its functionality as an A2A service. Your Customer Service Agent can then easily consume this service using ADK's `RemoteA2aAgent`.

```text
+-------------------------+         +-----------------------------------+
| Customer Service Agent  |         |         RemoteA2aAgent            |
| (Your Root Agent)       |<------->|         (ADK Client Proxy)        |
+-------------------------+         |                                   |
                                    |  +-----------------------------+  |
                                    |  |     Product Catalog Agent   |  |
                                    |  |      (External Service)     |  |
                                    |  +-----------------------------+  |
                                    +-----------------------------------+
                                                 |
                                                 | (Network Communication)
                                                 v
                                               +-----------------+
                                               |   A2A Server    |
                                               | (ADK Component) |
                                               +-----------------+
                                                       |
                                                       v
                                               +------------------------+
                                               | Product Catalog Agent  |
                                               | (Exposed Service)      |
                                               +------------------------+
```

- In this setup, first, the Product Catalog Agent needs to be exposed via an A2A Server. Then, the Customer Service Agent can simply call methods on the `RemoteA2aAgent` as if it were a tool, and the ADK handles all the underlying communication to the Product Catalog Agent. This allows for clear separation of concerns and easy integration of specialized agents.

- **There are two main ways to expose an ADK agent via A2A.**
  - **By using the `to_a2a(root_agent)` function:**
    - Use this function if you just want to convert an existing agent to work with A2A, and be able to expose it via a server through `uvicorn`, instead of `adk deploy api_server`.
    - This means that you have tighter control over what you want to expose via `uvicorn` when you want to productionize your agent.
    - Furthermore, the `to_a2a()` function auto-generates an agent card based on your agent code.
  - **By creating your own agent card(`agent.json`) and hosting it using `adk api_server --a2a`:**
    - There are two main benifits of using this approach. First, `adk api_server --a2a` works with `adk web`, making it easy to use, debug, and test your agent.
    - Second, with `adk api_server`, you can specify a parent folder with multiple, separate agents. Those agents that have an agent card (`agent.json`), will automatically be usable via A2A by other agents through the same server.
    - However, you will need to create your own agent cards. To create an agent card, you can follow the A2A Python Tutorial.

## **<font color="blue">Exposing the Remote Agent with the `to_a2a(root_agent)` Function</font>**
- You can take an existing agent built using the ADK and make it A2A-compatible by simply wrapping it using the `to_a2a()` function.
- The `to_a2a()` function will even auto-generate an **agent card** in memory behind-the-scenes by extracting  skills, capabilities, and metadata fro the ADK agent, so that the well-known agent card is made available when the agent endpoint is served using `uvicorn`.
- You can also provide your own agent card by using the agent_card parameter. The value can be an `AgentCard` object or a path to an agent card JSON file. 

In [ ]:
import os
from google.adk.agents.llm_agent import Agent
from google.adk.a2a.utils.agent_to_a2a import to_a2a
import uvicorn
from config import config


# -----------------------------
# Configuration
# -----------------------------
MODEL = config.MODEL
os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY.strip()


# -----------------------------
# Root Agent Definition
# -----------------------------
root_agent = Agent(
    model=MODEL,
    name="root_agent",
    description="Main assistant that interacts with users.",
    instruction="""
You are a helpful AI assistant.

Your responsibilities:
- Greet users
- Answer questions clearly
- Provide helpful explanations
- Assist with general tasks

Always respond in a friendly and professional way.
"""
)


# -----------------------------
# Convert Agent → A2A App
# -----------------------------
a2a_app = to_a2a(
    agent=root_agent,
    port=8001
)


# -------------
# Start A2A Server using Uvicorn
# -------------
if __name__ == "__main__":
    print("Starting A2A server (use CTRL+C to stop)…")
    uvicorn.run(
        a2a_app,
        host="0.0.0.0",
        port=8001,
        log_level="info"
    )


In [ ]:
# Root Agent A2A Server

import os
from google.adk.agents.llm_agent import Agent
from google.adk.a2a.utils.agent_to_a2a import to_a2a
from a2a.types import AgentCard
import uvicorn
from config import config


# ---------------------------------
# Configuration
# ---------------------------------
MODEL = config.MODEL
os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY.strip()


# ---------------------------------
# Root Agent Definition
# ---------------------------------
root_agent = Agent(
    model=MODEL,
    name="root_agent",
    description="Main assistant that interacts with users.",
    instruction="""
You are a helpful AI assistant.

Your responsibilities:
- Greet users
- Answer questions clearly
- Provide helpful explanations
- Assist with general tasks

Always respond in a friendly and professional way.
"""
)


# ---------------------------------
# A2A Agent Card
# ---------------------------------
agent_card = AgentCard(
    name="root_agent",
    url="http://localhost:8001",
    description="Root assistant agent exposed through A2A",
    version="1.0.0",
    capabilities={},
    skills=[],
    defaultInputModes=["text/plain"],
    defaultOutputModes=["text/plain"],
    supportsAuthenticatedExtendedCard=False,
)


# ---------------------------------
# Convert Agent → A2A Application
# ---------------------------------
a2a_app = to_a2a(
    agent=root_agent,
    port=8001,
    agent_card=agent_card
)


# -------------
# Start A2A Server using Uvicorn
# -------------
if __name__ == "__main__":
    print("Starting A2A server (use CTRL+C to stop)…")
    uvicorn.run(
        a2a_app,
        host="0.0.0.0",
        port=8001,
        log_level="info"
    )


In [ ]:
{
  "name": "root_agent",
  "url": "http://localhost:8001",
  "description": "Root assistant agent exposed through A2A",
  "version": "1.0.0",
  "capabilities": {},
  "skills": [
    {
      "id": "skill_general_chat",
      "name": "general_chat",
      "description": "General conversation and Q&A",
      "tags": ["chat", "qa"]
    }
  ],
  "defaultInputModes": ["text/plain"],
  "defaultOutputModes": ["text/plain"],
  "supportsAuthenticatedExtendedCard": false
}

In [ ]:
# Root Agent A2A Server

import os
from google.adk.agents.llm_agent import Agent
from google.adk.a2a.utils.agent_to_a2a import to_a2a
from config import config
from pathlib import Path
import sys
import uvicorn

# -----------------------------
# Configuration
# -----------------------------
MODEL = config.MODEL
os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY.strip()

# -----------------------------
# Root Agent Definition
# -----------------------------
root_agent = Agent(
    model=MODEL,
    name="root_agent",
    description="Main assistant that interacts with users.",
    instruction="""
You are a helpful AI assistant.

Your responsibilities:
- Greet users
- Answer questions clearly
- Provide helpful explanations
- Assist with general tasks

Always respond in a friendly and professional way.
"""
)

# -----------------------------
# Load A2A AgentCard from JSON
# -----------------------------
agent_card_path = Path(r"D:\Agent-Development-Kit\adk26\agent-card.json")

if not agent_card_path.is_file():
    print(f"Error: Agent card file not found: {agent_card_path}")
    sys.exit(1)

# -----------------------------
# Wrap Agent with A2A
# -----------------------------
a2a_app = to_a2a(
    agent=root_agent,
    port=8001,
    agent_card=str(agent_card_path)  # Path to JSON file
)

# -------------
# Start A2A Server using Uvicorn
# -------------
if __name__ == "__main__":
    print("Starting A2A server (use CTRL+C to stop)…")
    uvicorn.run(
        a2a_app,
        host="0.0.0.0",
        port=8001,
        log_level="info"
    )


In [ ]:
# Report Agent
import os
from google.adk.agents.llm_agent import Agent
from config import config

MODEL = config.MODEL
os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY.strip()

report_agent = Agent(
    model=MODEL,
    name="report_agent",
    description="Generates structured reports.",
    instruction="""
You are a professional report generation assistant.

Responsibilities:
- Generate structured reports
- Use headings and bullet points
- Keep explanations clear and concise
"""
)
